In [1]:
#!/usr/bin/env python3
"""
extract_prompts.py

Read a CSV, extract the "Prompt" column, normalize each prompt to a single line,
and write one prompt per line to a text file.

This variant uses constants (no command-line arguments).
Edit the INPUT_CSV / OUTPUT_TXT / PROMPT_COLUMN constants at the top as needed,
then run:

    python extract_prompts.py
"""
import csv
import os
import re
import sys

# --- Constants (edit these, don't pass as arguments) ---
INPUT_CSV = "dataset.csv"
OUTPUT_TXT = "rubend18.txt"
PROMPT_COLUMN = "Prompt"
# -------------------------------------------------------

def find_prompt_index(header):
    if not header:
        return None
    for i, col in enumerate(header):
        if col is None:
            continue
        if col.strip().lower() == PROMPT_COLUMN.strip().lower():
            return i
    return None

def normalize_to_single_line(s):
    if s is None:
        return ""
    return re.sub(r'\s+', ' ', s).strip()

def extract_prompts(infile, outfile):
    if not os.path.exists(infile):
        raise FileNotFoundError(f"Input file not found: {infile}")

    prompts = []
    with open(infile, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            header = []
        idx = find_prompt_index(header)
        # fallback: assume second column (index 1) if "Prompt" not found
        if idx is None:
            idx = 1 if len(header) > 1 else 0
            # include header row as data if header doesn't look like real header
            alpha_like = all((col is not None and re.match(r'^[A-Za-z _-]+$', col.strip())) for col in header if col)
            if not alpha_like and header:
                prompts.append(normalize_to_single_line(header[idx] if idx < len(header) else ""))

        for row in reader:
            val = row[idx] if idx < len(row) else ""
            prompts.append(normalize_to_single_line(val))

    with open(outfile, 'w', encoding='utf-8', newline='\n') as out:
        for p in prompts:
            out.write(p + "\n")

def main():
    try:
        extract_prompts(INPUT_CSV, OUTPUT_TXT)
        size = os.path.getsize(OUTPUT_TXT) if os.path.exists(OUTPUT_TXT) else 0
        print(f"Wrote prompts to {OUTPUT_TXT} ({size} bytes).")
    except Exception as e:
        print("Error:", e, file=sys.stderr)
        sys.exit(1)

if __name__ == "__main__":
    main()

Wrote prompts to rubend18.txt (159925 bytes).
